# Day 1 · Repeat the CPU exercise on a Colab GPU

Follow theory → doubts → practical → interview. This notebook is the **practical**, after the local CPU run. It has no precomputed outputs.

1. Select **Runtime → Change runtime type → GPU** if available.
2. Upload `day1_lifecycle.py` and the saved `cpu.json` when prompted.
3. Run the cells in order. We reuse the CPU model revision and settings.

Colab GPU availability varies; no paid runtime is required. Download your results before the temporary runtime ends. A single elapsed generation call is not a benchmark.


In [ ]:
import torch
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU allocated. Select a GPU runtime or report this run as unavailable.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
from google.colab import files
from pathlib import Path
import json

uploaded = files.upload()  # Select both files from your own Day 1 exercise.
required = {"day1_lifecycle.py", "cpu.json"}
if not required.issubset(uploaded):
    raise ValueError("Upload day1_lifecycle.py and cpu.json with exactly those names.")
cpu_record = json.loads(Path("cpu.json").read_text(encoding="utf-8"))
if cpu_record.get("track") != "cpu":
    raise ValueError("Expected the completed CPU generation record, not inventory-only output.")
print("Reusing revision:", cpu_record["model"]["revision"])
print("Prompt:", cpu_record["prompt"])


In [ ]:
import subprocess
import sys

# Match the small library packages to the CPU record. Keep Colab's CUDA PyTorch.
package_names = ["transformers", "huggingface-hub", "tokenizers", "safetensors", "psutil"]
requirements = [f"{name}=={cpu_record['hardware']['packages'][name]}" for name in package_names]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *requirements], check=True)
print("Installed matching packages; the GPU script runs in a fresh Python process.")


In [ ]:
subprocess.run([
    sys.executable, "day1_lifecycle.py",
    "--device", "cuda",
    "--revision", cpu_record["model"]["revision"],
    "--prompt", cpu_record["prompt"],
    "--max-new-tokens", str(cpu_record["settings"]["max_new_tokens"]),
    "--output", "gpu.json",
], check=True)
gpu_record = json.loads(Path("gpu.json").read_text(encoding="utf-8"))
for key in ["model", "dtype", "attention_implementation", "input_token_ids", "settings", "parameter_count", "logits_shape"]:
    print(key, "MATCH" if cpu_record[key] == gpu_record[key] else "DIFFERENT — inspect before comparing")
print("CPU device:", cpu_record["confirmed_model_device"])
print("GPU device:", gpu_record["confirmed_model_device"])
print("CPU generated tokens:", cpu_record["generated_token_count"])
print("GPU generated tokens:", gpu_record["generated_token_count"])
print("Output IDs equal:", cpu_record["generated_token_ids"] == gpu_record["generated_token_ids"])


In [ ]:
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
Path("gpu-environment.txt").write_text(freeze, encoding="utf-8")
files.download("gpu.json")
files.download("gpu-environment.txt")


## Discuss before the interview

Explain the prompt, input/generated token counts, logits shape, parameter count, fixed weights, and device. Inspect package and hardware differences. Matching greedy settings do not guarantee identical text across hardware.

The CPU pipeline is a later research exercise. No timing ratio, TTFT, or speedup is claimed here. A local CPU versus Colab GPU comparison also changes the host system.
